In [4]:

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [23]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')

openai = OpenAI()
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [24]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

## Training vs Inference time scaling

In [ ]:
## First we tried with small models and minimal reasoning and it failed. 
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/2

In [ ]:
## Then we increased the reasoning effort, it succeeded.  This is Inference scaling.
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

2/3

In [ ]:
## Next we took slightly more stronger model with minimum reasoning, again it succeeded. This is training scaling.
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

2/3

## Hard Challange

In [29]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [30]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

First, note the setup:
- Each volume has total page thickness of 2 cm (20 mm).
- Each cover thickness = 2 mm.
- The books are on a shelf in order: first volume (left) then second volume (right).
- The worm starts at the first page of the first volume and ends at the last page of the second volume, moving perpendicularly to the pages (i.e., horizontally through the stack).

Key points:
- For the first volume, the first page is on the inner side toward the second volume, and the outer cover is on the left.
- For the second volume, the last page is on the inner side toward the first volume, and the outer cover is on the right.
- The worm’s path is through the shared stack of pages and the outer pieces it must traverse from the very first page of volume 1 to the very last page of volume 2.

Compute the distance per side:
- In volume 1, to reach the first page from the left outer edge, the worm must pass through the outer cover (2 mm).
- In volume 2, to reach from the inner last page to the right outer edge, the worm must pass through the outer cover of volume 2 (2 mm).

Additionally, between volumes there is the inner joint where the two volumes touch:
- The inner faces of the two volumes touch each other; there is no physical material between the volumes at the seam to block the worm. The worm starts at the first page of volume 1 (the very inner face of its pages) and ends at the last page of volume 2 (the very inner face of those pages). The pages themselves in both volumes are part of the path, but the worm can go through the pages? The puzzle intends the worm gnaws a straight line from the first page of V1 to the last page of V2, only the thickness of covers and pages encountered along the way.

Total distance:
- Through the outer cover of volume 1: 2 mm
- Through the pages of volume 1 to reach the first page from the outer cover: but the worm starts at the first page, so it does not traverse any of volume 1 pages beyond that. It starts exactly at the first page, so no additional distance inside volume 1 beyond the initial outer cover.
- Across the seam between volumes: 0 mm (they touch directly).
- Across the inner pages of volume 2 from the inner last page to the outer cover on the right: similarly, the worm ends at the last page, so it does not traverse any of volume 2 pages beyond that inner page.
- Through the outer cover of volume 2: 2 mm

Sum: 2 mm + 2 mm = 4 mm.

Answer: 4 millimeters.

In [32]:
response = gemini.chat.completions.create(model="gemini-3.5-flash", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

The worm gnawed a distance of **4 mm** (or **0.4 cm**). 

Here is why:

When books stand on a shelf in order from left to right (Volume 1 on the left, Volume 2 on the right), their pages are oriented like this:

*   **Volume 1 (on the left):** The front cover is on the **right** side, and the back cover is on the left. Therefore, the *first page* of Volume 1 is on the far right of the book, right next to its front cover.
*   **Volume 2 (on the right):** The back cover is on the **left** side, and the front cover is on the right. Therefore, the *last page* of Volume 2 is on the far left of the book, right next to its back cover.

Since the two volumes are standing side-by-side, the sequence of components from left to right at the boundary where they meet is:

1.  First page of Volume 1
2.  **Front cover of Volume 1 (2 mm)**
3.  **Back cover of Volume 2 (2 mm)**
4.  Last page of Volume 2

To get from the first page of the first volume to the last page of the second volume, the worm only had to gnaw through the **front cover of Volume 1** and the **back cover of Volume 2**. 

$$\text{Total distance} = 2\text{ mm} + 2\text{ mm} = 4\text{ mm}$$

## Game Theory

In [33]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]

In [36]:
response = gemini.chat.completions.create(model="gemini-3.6-flash", messages=dilemma)
display(Markdown(response.choices[0].message.content))

I choose **Share**.

### The Rationale:

This scenario is a classic variation of the **Prisoner's Dilemma**. 

* **The Strict Game Theory Argument (for Stealing):** "Steal" is a weakly dominant strategy. If you share, you get $1,000 or $0. If you steal, you get $2,000 or $0. Strictly speaking, you never do worse by choosing "Steal." 

* **The Superrational Argument (for Sharing):** If I assume my partner is equally rational and analyzing the situation the exact same way I am, then whatever choice I make, they are likely to make too. 
  * If we both reason that "Steal" is smart, we both get **$0**.
  * If we both reason that "Share" leads to the best outcome for both of us, we both walk away with **$1,000**.

Because "Steal/Steal" results in mutually assured destruction ($0 for both), choosing **Share** is the only logical path toward securing a guaranteed positive payout for everyone, assuming a symmetric, rational partner.

In [37]:
response = openai.chat.completions.create(model="gpt-5.4-2026-03-05", messages=dilemma)
display(Markdown(response.choices[0].message.content))

Steal.